In [1]:
# [역할] 실습 환경 준비: 프로젝트 루트로 이동하고, 재료·모델 경로를 정하고, backend를 import 경로에 추가
import os        # 작업 폴더 변경용
import pathlib   # 경로를 객체로 다루기
import sys       # 모듈 검색 경로(sys.path) 조작용

here = pathlib.Path.cwd()                                                  # 현재 작업 폴더
# ROOT = next(p for p in (here, *here.parents) if (p / 'backend').is_dir())  # 위로 올라가며 backend 폴더가 있는 곳 = 프로젝트 루트
ROOT = here.parents[2] if here.name == "Tue" else here      # Fri에서 실행했으면 3단계 위가 프로젝트 루트
os.chdir(ROOT)                                              # 작업 폴더를 루트로 이동
SANDBOX = ROOT / "sandbox" / "W6" / "Tue"                   # 오늘 실습 폴더
SAMPLES = ROOT / "sandbox" / "W5" / "Mon" / "samples"       # 샘플 문서(재료) 폴더
MODELS = ROOT.parent / 'models'                             # 임베딩 모델 폴더 (프로젝트 옆)
FORMATS = SAMPLES / '_formats'                              # 여러 형식 샘플 폴더

print("프로젝트 루트  :", ROOT)       # 루트 경로 확인
print("재료 폴더      :", SAMPLES)    # 샘플 폴더 경로 확인

if str(ROOT / 'backend') not in sys.path:          # backend가 import 경로에 없으면
    sys.path.insert(0, str(ROOT / 'backend'))      # 맨 앞에 추가 → `from app...` import 가능

프로젝트 루트  : d:\hanwha-agent
재료 폴더      : d:\hanwha-agent\sandbox\W5\Mon\samples


In [ ]:
if not SAMPLES.exists():
    print('재료 폴더가 없습니다 :', SAMPLES)
    print('W5/Mon/samples 안 파일들을 확인해주세요.')
else:
    all_files = [p for p in SAMPLES.rglob('*') if p.is_file()]
    top_files = [p for p in SAMPLES.iterdir() if p.is_file()]
    fmt_files = [p for p in FORMATS.iterdir() if p.is_file()] if FORMATS.exists() else []
    print(f'재료 파일 : {len(all_files)}개')
    print(f'  samples 바로 아래 : {len(top_files)}개 · _formats 안 : {len(fmt_files)}개')
    print('오늘 쓰는 재료 :')
    for p in sorted(SAMPLES.glob('DOC-FI-009*')):
        print(f'  {p.name}')

재료 파일 : 31개
  samples 바로 아래 : 24개 · _formats 안 : 7개
오늘 쓰는 재료 :
  DOC-FI-009_법인카드_사용_지침_v1.4.docx
  DOC-FI-009_법인카드_사용_지침_v1.4.pdf


In [3]:
from app.rag import chunker, parser

p = next(SAMPLES.glob('DOC-FI-009_*_v1.4.docx'))
doc = parser.parse(p)
print(f'블록 {len(doc.blocks)} · 표 {doc.table_count} · 쪽 {doc.page_count}')
drafts = chunker.chunk(doc)
print(chunker.summarize(drafts))
for i, d in enumerate(drafts):
    if d.kind == '표':
        print(f'  [표] ord={i}  locator={d.locator}  글자 수={len(d.text)}')
print('★ 이 drafts 리스트는 이 셀이 끝나면 사라집니다. 담을 자리가 없습니다.')

d:\hanwha-agent\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


블록 81 · 표 2 · 쪽 1
조항 단위 17 + 표 단위 2 = 19 청크
  [표] ord=1  locator=표1  글자 수=264
  [표] ord=18  locator=표2  글자 수=283
★ 이 drafts 리스트는 이 셀이 끝나면 사라집니다. 담을 자리가 없습니다.


In [5]:
import importlib
import app.models

importlib.reload(app.models)

print('Base.metadata 에 등록된 표:', sorted(app.models.Base.metadata.tables))

Base.metadata 에 등록된 표: ['chunks', 'departments', 'document_versions', 'documents', 'run_steps', 'runs', 'usage_logs', 'users']


In [6]:
import importlib

UPLOADED = "uploads/DOC-FI-009_v1.4.docx" 

from app.services import document_service

importlib.reload(document_service)
print(document_service.ingest_document(doc_id="DOC-FI-009", version="v1.4", path=UPLOADED))

NoForeignKeysError: Could not determine join condition between parent/child tables on relationship DocumentVersion.chunks - there are no foreign keys linking these tables.  Ensure that referencing columns are associated with a ForeignKey or ForeignKeyConstraint, or specify a 'primaryjoin' expression.

In [7]:
import time
from app.rag import chunker, parser

PATTERNS = ['DOC-FI-009_*_v1.4.docx', 'DOC-FI-009_*_v1.4.pdf', 'DOC-HR-014_*_v2.0.docx', 'DOC-HR-014_*_v2.0.pdf']

for pattern in PATTERNS:
    path = next(SAMPLES.glob(pattern))
    started = time.perf_counter()
    parsed = parser.parse(path)
    drafts = chunker.chunk(parsed)
    elapsed = time.perf_counter() - started
    print(f'{path.name:<34} {len(drafts):>3}청크   {elapsed:.3f}초')
TIMEOUT = 10.0
print()
print(f'TIMEOUT = {TIMEOUT} 초 — 우리 재료는 전부 그 아래입니다.')


DOC-FI-009_법인카드_사용_지침_v1.4.docx     19청크   0.062초
DOC-FI-009_법인카드_사용_지침_v1.4.pdf      17청크   0.381초
DOC-HR-014_국내출장_여비_규정_v2.0.docx     53청크   0.054초
DOC-HR-014_국내출장_여비_규정_v2.0.pdf      45청크   0.346초

TIMEOUT = 10.0 초 — 우리 재료는 전부 그 아래입니다.
